<a href="https://colab.research.google.com/github/ajtamayoh/In2Lab-TNT-at-SMM4H-2026/blob/main/code_subtask_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In2Lab-TNT at #SMM4H-HeaRD 2026: An Application of QTT's Terminological Entanglement to Leverage Insomnia Detection in Clinical Notes

Antonio Tamayo-Herrera (University of Antioquia)

Giovanny Díaz-Laínes (University of Antioquia)

Carlos Mario Pérez-Pérez (National Autonomous University of Mexico) (UNAM)

Diego A. Burgos Wake Forest (University)

Correspondence: antonio.tamayo@udea.edu.co

# 1. Installation and Data Preparation

First, we merge the CSV with the new JSON format from Subtask 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import json
import os
from openai import OpenAI
from sklearn.metrics import f1_score, classification_report

# Configuration
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

In [ ]:
# Validation

# 1. Load the corpus
df_corpus = pd.read_csv('validation/corpus_validation.csv')

# 2. Load the labels (Subtask 2)
with open('validation/subtask_2.json', 'r') as f:
    labels_2_dict = json.load(f)

# Function to flatten the Subtask 2 JSON and obtain the final label (Insomnia Status)
def get_final_status(row_data):
    # Rule: Rule A (Def1 AND Def2) OR Rule B OR Rule C
    def1 = row_data.get('Definition 1', {}).get('label', 'no') == 'yes'
    def2 = row_data.get('Definition 2', {}).get('label', 'no') == 'yes'
    rule_a = 'yes' if (def1 and def2) else 'no'
    rule_b = row_data.get('Rule B', {}).get('label', 'no')
    rule_c = row_data.get('Rule C', {}).get('label', 'no')

    if rule_a == 'yes' or rule_b == 'yes' or rule_c == 'yes':
        return 'yes'
    return 'no'

# Prepare training/validation dataset
data_rows = []
for note_id, info in labels_2_dict.items():
    data_rows.append({
        'note_id': int(note_id),
        'def1_gold': info.get('Definition 1', {}).get('label', 'no'),
        'def2_gold': info.get('Definition 2', {}).get('label', 'no'),
        'rule_b_gold': info.get('Rule B', {}).get('label', 'no'),
        'rule_c_gold': info.get('Rule C', {}).get('label', 'no'),
        'insomnia_status_gold': get_final_status(info)
    })

df_labels_2 = pd.DataFrame(data_rows)
df_final = pd.merge(df_corpus, df_labels_2, on='note_id')

In [ ]:
#Testing

# 1. Load the corpus
df_corpus = pd.read_csv('testing/corpus_testing.csv')
df_final = df_corpus.copy()

In [ ]:
df_final.head(10)

In [ ]:
df_final.shape

# 2. Prompt Configuration "Expert Clinical Reasoner"

For this task, max_tokens must be higher (aprox. 500) because the model must generate explanations. We will use few-shot examples within the System Prompt to ensure that the output JSON is perfectly formatted.

In [ ]:
SYSTEM_PROMPT = """You are a clinical expert. Analyze the provided clinical note and identify insomnia criteria.

RULES:
- Def 1: Difficulty initiating/maintaining sleep or explicit insomnia.
- Def 2: Daytime impairment (fatigue, irritability, etc.) or explicit insomnia.
- Rule A: yes if Def 1 AND Def 2 are yes.
- Rule B: yes if taking: Estazolam, Eszopiclone, Flurazepam, Lemborexant, Quazepam, Ramelteon, Suvorexant, Temazepam, Triazolam, Zaleplon, Zolpidem.
- Rule C: yes if taking: Alprazolam, Clonazepam, Lorazepam, Melatonin, Quetiapine, Trazodone (and others) AND has symptoms of Def 1 or Def 2.
- Insomnia Status: yes if Rule A, B, or C is yes.

Mandatory constraints:

The 'explanation' field must be a literal string extracted from the text (copy and paste) to justify the label. This process must be case sensitive. If there is a spelling error, it must be copied exactly as it appears. For example, a space before a period (increased .) or a missing space after a period, such as (well.pt anxious).

Do not add any extra character at the end of the explanation. For example, if there is not a period at the end, do not add it.

If the explanation appears in a non-contiguous text, separate the terms with ' | '. For example: ... Lorazepam..., ... Clonazepam... , ... Quetiapine... The text must be extracted as Lorazepam | Clonazepam | Quetiapine

If a component is labeled "yes", the explanation must be non-empty. If a component is labeled "no", the explanation must be empty.

Return ONLY a JSON object with this structure:

{
  "definition_1": {"label": "yes/no", "explanation": "literal text from note"},
  "definition_2": {"label": "yes/no", "explanation": "literal text from note"},
  "rule_a": {"label": "yes/no"},
  "rule_b": {"label": "yes/no", "explanation": "medication name1 | medication name2 | medication name3"},
  "rule_c": {"label": "yes/no", "explanation": "medication | Def 1 or Def 2 symptoms extracted as literal string in the text"},
  "insomnia_status": {"label": "yes/no"}
}

"""

def analyze_insomnia_complex(text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analyze this note: {text}"}
            ],
            response_format={ "type": "json_object" },
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return None

# 3. Execution and Inference

We process the data and extract the labels to compute the evaluation metrics.

In [ ]:
from openai import RateLimitError
import time

def safe_call(func, *args, **kwargs):
    delay = 0.001  # initial 1 ms

    while True:
        try:
            return func(*args, **kwargs)

        except RateLimitError:
            time.sleep(delay)
            delay = min(delay * 2, 1)  # exponential backoff up to 1 second

        except Exception as e:
            # optional: raise other errors
            raise e


results = []

# We process the dataframe
for index, row in df_final.iterrows():
    prediction = safe_call(analyze_insomnia_complex, row['text'])

    if prediction:
        # Extract each field from the JSON returned by GPT
        results.append({
            'note_id': row['note_id'],
            'pred_status': prediction.get('insomnia_status', {}).get('label', 'no'),

            # Definition 1
            'pred_def1': prediction.get('definition_1', {}).get('label', 'no'),
            'exp_def1': prediction.get('definition_1', {}).get('explanation', ''),

            # Definition 2
            'pred_def2': prediction.get('definition_2', {}).get('label', 'no'),
            'exp_def2': prediction.get('definition_2', {}).get('explanation', ''),

            # Rule B
            'pred_rule_b': prediction.get('rule_b', {}).get('label', 'no'),
            'exp_rule_b': prediction.get('rule_b', {}).get('explanation', ''),

            # Rule C
            'pred_rule_c': prediction.get('rule_c', {}).get('label', 'no'),
            'exp_rule_c': prediction.get('rule_c', {}).get('explanation', '')
        })

# Create the dataframe with ALL columns
df_results = pd.DataFrame(results)
print("Generated columns:", df_results.columns.tolist())
df_results.head()

In [ ]:
# Save results

#df_results.to_csv('validation/subtask_2/subtask_2.csv', index=False)

df_results.to_csv('testing/subtask_2/subtask_2.csv', index=False)

## Convert the data into the competition JSON format and save it to Google Drive

In [ ]:
import json
import os

def find_span(target_text, full_text):
    """Busca el inicio y fin del texto en la nota original."""
    target = target_text.split("|")
    if target:
      o = ""
      for t in target:
        start = full_text.strip().lower().find(t.strip().lower())
        if start != -1:
            end = start + len(t)
            o+=f"{start} {end};"
      return [o[:-1]]
    else:
      if not target_text or target_text.lower() == "none" or len(target_text) < 2:
        return ["0 0"] # To avoid format issues and keep possible good predictions

      start = full_text.strip().lower().find(target_text.strip().lower())
      if start != -1:
          end = start + len(target_text)
          return [f"{start} {end}"]
      return []

def export_to_json_format(df_results, df_full_corpus):
    """
    Convierte el dataframe de resultados al formato oficial de la Subtask 2.
    """
    output_dict = {}

    for _, row in df_results.iterrows():
        note_id = str(row['note_id'])
        # Retrieve the original note text to compute spans
        original_note = df_full_corpus[df_full_corpus['note_id'] == int(note_id)]['text'].values[0]

        # Structure for each note
        output_dict[note_id] = {
            "Definition 1": {
                "label": row['pred_def1'],
                "span": find_span(row['exp_def1'], original_note) if row['pred_def1'] == 'yes' else [],
                #"text": [row['exp_def1']] if row['pred_def1'] == 'yes' else [],
                #"original": original_note,
            },
            "Definition 2": {
                "label": row['pred_def2'],
                "span": find_span(row['exp_def2'], original_note) if row['pred_def2'] == 'yes' else [],
                #"text": [row['exp_def2']] if row['pred_def2'] == 'yes' else [],
                #"original": original_note,
            },
            # Note: We add Rule B and C following the same dataframe logic
            "Rule B": {
                "label": row.get('pred_rule_b', 'no'),
                "span": find_span(row.get('exp_rule_b', ''), original_note) if row.get('pred_rule_b') == 'yes' else [],
                #"text": [row.get('exp_rule_b', '')] if row.get('pred_rule_b') == 'yes' else [],
                #"original": original_note,
            },
            "Rule C": {
                "label": row.get('pred_rule_c', 'no'),
                "span": find_span(row.get('exp_rule_c', ''), original_note) if row.get('pred_rule_c') == 'yes' else [],
                #"text": [row.get('exp_rule_c', '')] if row.get('pred_rule_c') == 'yes' else [],
                #"original": original_note,
            }
        }

    return output_dict

In [ ]:
# 2. Generate the final dictionary
final_json_data = export_to_json_format(df_results, df_corpus)

# 3. Save to Google Drive

#Validation
#file_path = 'predictions/validation/subtask_2/subtask_2.json'

# Testing
file_path = 'predictions/testing/subtask_2/subtask_2.json'

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(final_json_data, f, indent=4, ensure_ascii=False)

print(f"File saved at: {file_path}")

# 4. Metric Evaluation (Micro-F1)

Since this is a multilabel task, Micro-F1 is the standard metric. Here we compare the "Insomnia Status".

In [ ]:
# Compare against the Gold Standard (Status Final)
y_true = df_final.head(len(df_results))['insomnia_status_gold']
y_pred = df_results['pred_status']

print(f"F1 Score (Micro): {f1_score(y_true, y_pred, average='micro'):.4f}")
print("\nReport by class:")
print(classification_report(y_true, y_pred))

#5. Inference with Explanations for New Texts

This function allows you to provide a string and inspect the model's clinical reasoning.

In [ ]:
def predict_and_explain(text):
    res = analyze_insomnia_complex(text)
    print("--- ANALYSIS ---")
    print(f"Status Final: {res['insomnia_status']['label'].upper()}")
    print("-" * 30)
    print(f"Def 1 (Sleep Difficulty): {res['definition_1']['label']}")
    print(f"Evidence: {res['definition_1']['explanation']}")
    print(f"Def 2 (Daytime Impairment): {res['definition_2']['label']}")
    print(f"Evidence: {res['definition_2']['explanation']}")
    print(f"Rule B (Primary Medication): {res['rule_b']['label']}")
    return res

# Test
test_note = "Patient reports waking up at 3 AM every night and feeling very fatigued during work. No medications prescribed."
predict_and_explain(test_note)

# ROUGE

Technical Details of ROUGE:

ROUGE-1: Measures how many individual words overlap. It is excellent for assessing whether the model captured key terms (como "insomnia", "fatigue").

ROUGE-2: Measures word pairs. It better reflects whether the model is reproducing exact phrases from the clinical note.

ROUGE-L: It measures the longest common sequence. This is very useful for assessing whether the structure of the explanation is similar to that of the expert.

In [ ]:
# 1. Installation of the required library
!pip install evaluate rouge_score

In [ ]:
import evaluate

# Load the ROUGE metric
rouge = evaluate.load('rouge')

def calculate_rouge_multi(df_results, labels_2_dict):
    all_references = []
    all_predictions = []

    # Mapping of dataframe columns vs. original JSON keys
    fields = {
        'exp_def1': 'Definition 1',
        'exp_def2': 'Definition 2',
        'exp_rule_b': 'Rule B',
        'exp_rule_c': 'Rule C'
    }

    for index, row in df_results.iterrows():
        note_id_str = str(row['note_id'])
        if note_id_str in labels_2_dict:
            for df_col, json_key in fields.items():
                # Retrieve the gold reference
                gold_texts = labels_2_dict[note_id_str].get(json_key, {}).get('text', [])
                ref_text = " ".join(gold_texts)

                # Model prediction
                pred_text = row.get(df_col, "")

                # We only compare when both contain content to avoid bias from zeros
                if ref_text.strip() and pred_text.strip():
                    all_references.append(ref_text)
                    all_predictions.append(pred_text)

    if not all_predictions:
        print("There are not enough text pairs to evaluate ROUGE.")
        return None

    results = rouge.compute(predictions=all_predictions, references=all_references)

    print("--- GLOBAL ROUGE METRICS (All Definitions) ---")
    for key, value in results.items():
        print(f"{key.upper()}: {value:.4f}")

    return results

# Run
rouge_global = calculate_rouge_multi(df_results, labels_2_dict)

# BERTScore

It is ideal for this task because, unlike ROUGE (which looks for exact word matches), it uses embeddings to determine whether the semantic meaning is equivalent. If the patient says "extreme exhaustion" and the model extracts "severe fatigue," ROUGE would give a low score, but BERTScore would recognize that they are semantically identical.

In [ ]:
# 1. Installation of the BERTScore library
!pip install bert_score

In [ ]:
from bert_score import score

def calculate_bertscore_multi(df_results, labels_2_dict):
    all_references = []
    all_predictions = []

    fields = {
        'exp_def1': 'Definition 1',
        'exp_def2': 'Definition 2',
        'exp_rule_b': 'Rule B',
        'exp_rule_c': 'Rule C'
    }

    for index, row in df_results.iterrows():
        note_id_str = str(row['note_id'])
        if note_id_str in labels_2_dict:
            for df_col, json_key in fields.items():
                gold_texts = labels_2_dict[note_id_str].get(json_key, {}).get('text', [])
                ref_text = " ".join(gold_texts)
                pred_text = row.get(df_col, "")

                if ref_text.strip() and pred_text.strip():
                    all_references.append(ref_text)
                    all_predictions.append(pred_text)

    if not all_predictions:
        print("There is no data for BERTScore.")
        return None

    P, R, F1 = score(all_predictions, all_references, lang="en", verbose=False)

    print("\n--- GLOBAL BERTSCORE METRICS ---")
    print(f"Precision: {P.mean():.4f}")
    print(f"Recall:    {R.mean():.4f}")
    print(f"F1-Score:  {F1.mean():.4f}")

    return {"F1": F1.mean()}

# Run
bert_global = calculate_bertscore_multi(df_results, labels_2_dict)